In [ ]:
import rasterio
import numpy as np
import os
import gc
from tqdm import tqdm

# Input directories
disturbance_dir = r'G:\Hangkai\CONUS_Forest_Edge_LCMAP\Disturbance_Map'
edge_dynamic_dir = r'G:\Hangkai\CONUS_Forest_Edge_LCMAP\LCMAP_edge_dynamics'
output_dir = r'G:\Hangkai\CONUS_Forest_Edge_LCMAP\dynamics_with_disturbance'

# Ensure the output directory exists
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Year range to process
years = range(2010, 2021)  # From 1985 to 2020

# Process each year
for year in tqdm(years):
    # Construct file paths for disturbance and edge dynamics maps
    disturbance_file = os.path.join(disturbance_dir, f'CONUS_Disturbance_{year}.tif')
    edge_dynamic_file = os.path.join(edge_dynamic_dir, f'LCMAP_{year}_{year+1}_edge_dynamics.tif')

    # Check if both files exist
    if not os.path.exists(disturbance_file) or not os.path.exists(edge_dynamic_file):
        print(f"Missing file: {disturbance_file} or {edge_dynamic_file}")
        continue

    # Open disturbance map
    with rasterio.open(disturbance_file) as disturbance_src:
        disturbance_data = disturbance_src.read(1)  # Read the first band (assuming single-band)
        profile = disturbance_src.profile  # Get profile for output purposes

    # Process each band (representing different edges) in the edge dynamics map
    for band_index in range(1, 5):
        with rasterio.open(edge_dynamic_file) as edge_src:
            edge_dynamic_data = edge_src.read(band_index)  # Read the specified band

        # Compute: edge dynamic value * 10 + disturbance value
        combined_data = edge_dynamic_data * 10 + disturbance_data
        
        del edge_dynamic_data
        gc.collect()


        # Define the output file path
        output_path = os.path.join(output_dir, f'output_combined_{year}_band_{band_index}.tif')

        # Save the result
        profile.update(dtype=rasterio.int8)  # Update profile to match data type
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(combined_data.astype(rasterio.int8), 1)

        print(f"Processing complete: {output_path}")
        del combined_data
        gc.collect()

print("All bands processed for all years!")
